# 第 21 章　波动率建模：局部、随机、跳跃与粗糙

本章出现的每一个模型，都从它自己的定义搭起来，再让它回答本章提出的问题。
**只用 NumPy 与 SciPy。** Heston 的特征函数、Dupire 公式、Hagan 展开、
Merton 级数以及粗糙 Bergomi 的混合格式都是逐条写出来的，而不是从库里调用，
于是正文里的方程与代码一一对应。

| 节 | 搭什么 | 方法 | 正文 |
|---|---|---|---|
| 1 | 布莱克–斯科尔斯–默顿公式与隐含波动率 | 闭式解、Brent 求根 | — |
| 2 | Heston | 特征函数、Gauss–Legendre 积分 | (21.4) |
| 3 | 每个参数各管什么 | 微笑及其拟合统计量 | 图 21.2、表 21.2 |
| 4 | 局部波动率 | Dupire，用有限差分 | (21.2)、例 21.1 |
| 5 | 前向微笑 | 局部波动率函数自身的偏斜 | 例 21.2 |
| 6 | 偏斜的期限结构 | Heston 对幂律 | 图 21.4 |
| 7 | SABR | Hagan 展开，以及它的密度在哪里变负 | 第 21.4 节，(29.21)–(29.22) |
| 8 | 跳跃 | Merton 级数 | 例 21.3 |
| 9 | 粗糙波动率 | 粗糙 Bergomi，混合格式 | (21.8)、(21.9) |
| 10 | 局部–随机波动率 | 用粒子法求杠杆函数 | (21.10)、例 21.5、图 21.6 |
| 11 | 模型风险 | 共同随机数下的障碍期权 | 例 21.7 |
| 12 | 校准目标函数 | 同一曲面的两种拟合 | 例 21.6 |

全篇取 `S0 = 100`、`r = q = 0`，基准情形即本章的
`v0 = theta = 0.04`、`kappa = 1.5`、`sigma_v = 0.6`、`rho = -0.7`。

蒙特卡洛结果一律按**标准误差**比较，绝不逐位比较。

表头保留英文：中文字符在等宽终端里是双宽的，而 Python 的 `%-10s` 按字符数计。

## 1. 布莱克–斯科尔斯–默顿公式与隐含波动率

下面所有结果都以隐含波动率报出，所以反解必须先写。
对价格用 Brent 求根比 Newton 迭代慢，但绝对可靠；在 notebook 里，这个取舍是对的。

In [1]:
import numpy as np
from math import log, exp, sqrt, pi, factorial
from scipy.stats import norm
from scipy.optimize import brentq, least_squares

S0, R, Q = 100.0, 0.0, 0.0
BASE = dict(v0=0.04, kappa=1.5, theta=0.04, sigv=0.6, rho=-0.7)


def bs_call(S, K, r, q, sig, T):
    if sig <= 0 or T <= 0:
        return max(S * exp(-q * T) - K * exp(-r * T), 0.0)
    d1 = (log(S / K) + (r - q + 0.5 * sig * sig) * T) / (sig * sqrt(T))
    d2 = d1 - sig * sqrt(T)
    return S * exp(-q * T) * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)


def bs_vega(S, K, r, q, sig, T):
    d1 = (log(S / K) + (r - q + 0.5 * sig * sig) * T) / (sig * sqrt(T))
    return S * exp(-q * T) * norm.pdf(d1) * sqrt(T)


def implied_vol(price, S, K, r, q, T):
    """Brent between 0.01% and 500% volatility."""
    lo, hi = 1e-6, 5.0
    if price <= bs_call(S, K, r, q, lo, T) + 1e-14:
        return float("nan")
    return brentq(lambda s: bs_call(S, K, r, q, s, T) - price, lo, hi, xtol=1e-12)


print("往返检验：给一年期 20% 波动率的看涨期权定价，再反解回来")
p = bs_call(S0, 100.0, R, Q, 0.20, 1.0)
print("  价格 %.6f   反解回 %.8f" % (p, implied_vol(p, S0, 100.0, R, Q, 1.0)))

往返检验：给一年期 20% 波动率的看涨期权定价，再反解回来
  价格 7.965567   反解回 0.20000000


## 2. Heston

式 (21.4) 是一对相关的扩散过程，

$$dS = (r-q)S\,dt + \sqrt{v}\,S\,dW_1, \qquad
  dv = \kappa(\theta-v)\,dt + \sigma_v\sqrt{v}\,dW_2, \qquad dW_1dW_2 = \rho\,dt$$

它在实际使用中的全部吸引力在于 $\log S_T$ 的特征函数是现成的，
于是定价只需一个数值积分，而不是一次模拟：

$$C = S_0e^{-qT}P_1 - Ke^{-rT}P_2, \qquad
  P_j = \tfrac12 + \frac{1}{\pi}\int_0^\infty
  \mathrm{Re}\!\left[\frac{e^{-iu\ln K}\varphi_j(u)}{iu}\right]du$$

有两处实现细节不能省。特征函数里的平方根必须取能让
$\log((1-ge^{-dT})/(1-g))$ 保持连续的分支，以规避
*小 Heston 陷阱*；下面写的是稳定的形式。
另外被积函数光滑且衰减很快，于是 $[0, 150]$ 上一条固定的
**Gauss–Legendre** 求积公式可以一次算出一整排执行价，并且精确到机器精度，
这正是第 4 节里 Dupire 差分得以成立的前提。

真正关键的检验只有一个：当 $\sigma_v \to 0$，模型必须退化为
波动率取 $\sqrt{v_0}$ 的布莱克–斯科尔斯–默顿模型。

In [2]:
GL_N, UMAX = 512, 150.0
_u, _w = np.polynomial.legendre.leggauss(GL_N)
U = 0.5 * (_u + 1.0) * UMAX
W = _w * 0.5 * UMAX


def heston_cf(u, T, p):
    """E[exp(i u log S_T)] on the continuous branch."""
    iu = 1j * u
    b = p["kappa"] - p["rho"] * p["sigv"] * iu
    d = np.sqrt(b * b + p["sigv"] ** 2 * (iu + u * u))
    g = (b - d) / (b + d)
    e = np.exp(-d * T)
    C = (p["kappa"] * p["theta"] / p["sigv"] ** 2) * (
        (b - d) * T - 2.0 * np.log((1 - g * e) / (1 - g)))
    D = (p["v0"] / p["sigv"] ** 2) * (b - d) * (1 - e) / (1 - g * e)
    return np.exp(iu * (log(S0) + (R - Q) * T) + C + D)


def heston_call(Ks, T, **kw):
    """Vectorized over strikes. Returns an array."""
    p = dict(BASE); p.update(kw)
    Ks = np.atleast_1d(np.asarray(Ks, float))
    lnK = np.log(Ks)[:, None]
    fwd = S0 * exp((R - Q) * T)
    e = np.exp(-1j * U * lnK)
    P1 = 0.5 + (W * np.real(e * heston_cf(U - 1j, T, p) / (1j * U * fwd))).sum(1) / pi
    P2 = 0.5 + (W * np.real(e * heston_cf(U, T, p) / (1j * U))).sum(1) / pi
    return S0 * exp(-Q * T) * P1 - Ks * exp(-R * T) * P2


def heston_iv(K, T, **kw):
    return implied_vol(float(heston_call(K, T, **kw)[0]), S0, float(K), R, Q, T)


print("极限检验：sigma_v -> 0 必须给出波动率为 sqrt(v0) = 20% 的布莱克–斯科尔斯–默顿价格")
print("  Heston   %.10f" % heston_call(100.0, 1.0, sigv=1e-6, rho=0.0)[0])
print("  BS(20%%)  %.10f" % bs_call(S0, 100.0, R, Q, 0.20, 1.0))
print()
print("基准情形，一年期")
print("  平值看涨 %.4f    平值隐含波动率 %.2f%%"
      % (heston_call(100.0, 1.0)[0], 100 * heston_iv(100.0, 1.0)))

极限检验：sigma_v -> 0 必须给出波动率为 sqrt(v0) = 20% 的布莱克–斯科尔斯–默顿价格
  Heston   7.9655612546
  BS(20%)  7.9655674554

基准情形，一年期
  平值看涨 6.7509    平值隐含波动率 16.94%


## 3. 每个参数各管什么：图 21.2 与表 21.2

两个统计量就能概括一条微笑，二者都以对数价值状态
$k = \log(K/F)$ 度量，在 $k = \pm 0.10$ 处取中心差分：

* **偏斜** $\;\partial\sigma/\partial k$：倾角，主要由 $\rho$ 决定；
* **曲率** $\;\partial^2\sigma/\partial k^2$：弯度，主要由 $\sigma_v$ 决定。

这张表的要点在于*相互作用*。调高 $\sigma_v$ 并不只是抬高曲率：
它同时会让偏斜变陡、让平值水平下降，
所以校准是一次真正的优化，而不是三次一维拟合。

In [3]:
KS_FIG = np.array([70., 80., 90., 100., 110., 120., 130.])

print("图 21.2，一年期隐含波动率 (%)")
print("  %-16s %s" % ("", "".join("%8.0f" % K for K in KS_FIG)))
for tag, kw in [("rho=-0.9", dict(rho=-0.9)), ("rho=-0.5", dict(rho=-0.5)),
                ("rho=0", dict(rho=0.0)), ("sigv=0.2", dict(sigv=0.2)),
                ("sigv=0.6 (基准)", {}), ("sigv=1.0", dict(sigv=1.0))]:
    row = [100 * heston_iv(K, 1.0, **kw) for K in KS_FIG]
    print("  %-16s %s" % (tag, "".join("%8.2f" % x for x in row)))


def smile_stats(T=1.0, h=0.10, **kw):
    """ATM level, skew and curvature in log-moneyness, central differences at +/- h."""
    F = S0 * exp((R - Q) * T)
    a, b, c = (heston_iv(F * exp(-h), T, **kw), heston_iv(F, T, **kw),
               heston_iv(F * exp(h), T, **kw))
    return 100 * b, (c - a) / (2 * h), (c - 2 * b + a) / h ** 2


print()
print("表 21.2，一年期")
print("  %-28s %9s %9s %11s" % ("", "ATM (%)", "Skew", "Curvature"))
for tag, kw in [("基准", {}), ("rho = -0.9", dict(rho=-0.9)), ("rho = 0", dict(rho=0.0)),
                ("sigma_v = 0.2", dict(sigv=0.2)), ("sigma_v = 1.0", dict(sigv=1.0)),
                ("kappa = 5", dict(kappa=5.0)), ("theta = 0.09", dict(theta=0.09))]:
    a, b, c = smile_stats(**kw)
    print("  %-28s %9.2f %9.3f %11.3f" % (tag, a, round(b, 3) + 0.0, c))

图 21.2，一年期隐含波动率 (%)
                         70      80      90     100     110     120     130
  rho=-0.9            27.52   23.98   20.33   16.46   12.39    9.43    8.79
  rho=-0.5            25.70   22.67   19.79   17.32   15.86   15.71   16.27
  rho=0               22.69   20.39   18.61   17.91   18.49   19.71   21.06
  sigv=0.2            23.24   21.88   20.61   19.44   18.37   17.42   16.61
  sigv=0.6 (基准)       26.66   23.37   20.10   16.94   14.39   13.33   13.44
  sigv=1.0            27.55   23.16   18.76   14.45   11.79   12.16   13.29

表 21.2，一年期
                                 ATM (%)      Skew   Curvature
  基准                               16.94    -0.282       0.356


  rho = -0.9                       16.46    -0.398      -0.581
  rho = 0                          17.91     0.000       1.268
  sigma_v = 0.2                    19.44    -0.112      -0.011
  sigma_v = 1.0                    14.45    -0.340       1.396
  kappa = 5                        18.94    -0.155       0.031
  theta = 0.09                     22.70    -0.227       0.084


## 4. 局部波动率与“二倍法则”：例 21.1

Dupire 公式 (21.2) 直接从看涨期权曲面上读出唯一的局部波动率函数：

$$\sigma_{\rm loc}^2(K,T) = \frac{\partial C/\partial T + (r-q)K\,\partial C/\partial K + qC}
                                    {\tfrac12 K^2\,\partial^2C/\partial K^2}$$

分母是 $\tfrac12K^2$ 乘以隐含密度，所以整个式子不过是曲面已经含有的两个量之比。

从数值上看它是期权价格的二阶导数，第 2 节那条 Gauss–Legendre 求积之所以
值得费事，正是因为这一点：价格精确到约 $10^{-9}$，于是对数执行价上 1% 的扰动远高于噪声底。
在 $\log K$ 而不是 $K$ 上作差分，是它在两翼仍然稳定的关键：
取 $y = \log K$ 时 $K^2\partial^2C/\partial K^2 = C_{yy} - C_y$。

结果就是**二倍法则**：局部波动率曲线的陡度大约是导出它的隐含波动率曲线的两倍。
这不是校准失败；隐含波动率是沿途各个局部波动率的*平均*，而平均会把曲线抹平。

In [4]:
def local_vol(Ks, T, hk=0.01, hT=0.005, **kw):
    """Dupire by central differences in log-strike and in maturity."""
    Ks = np.atleast_1d(np.asarray(Ks, float))
    y = np.log(Ks)
    Tm = max(T - hT, 1e-4)
    C0 = heston_call(Ks, T, **kw)
    dCdT = (heston_call(Ks, T + hT, **kw) - heston_call(Ks, Tm, **kw)) / (T + hT - Tm)
    Cp, Cm = heston_call(np.exp(y + hk), T, **kw), heston_call(np.exp(y - hk), T, **kw)
    Cy = (Cp - Cm) / (2 * hk)
    Cyy = (Cp - 2 * C0 + Cm) / hk ** 2
    dens = Cyy - Cy                      # = K^2 d2C/dK2
    num = dCdT + (R - Q) * Cy + Q * C0
    return np.sqrt(np.clip(num / np.maximum(0.5 * dens, 1e-14), 1e-4, 4.0))


KS_EX = np.array([80., 90., 95., 100., 105., 110., 120.])
iv = np.array([heston_iv(K, 1.0) for K in KS_EX])
lv = local_vol(KS_EX, 1.0)

print("例 21.1：一年期，S0 = 100")
print("  %8s %14s %14s" % ("Strike", "Implied vol", "Local vol"))
for K, a, b in zip(KS_EX, iv, lv):
    print("  %8.0f %13.2f%% %13.2f%%" % (K, 100 * a, 100 * b))

k = log(1.05) - log(0.95)
s_imp = (iv[4] - iv[2]) / k
s_loc = (lv[4] - lv[2]) / k
print()
print("  对数执行价上的斜率，在 95 与 105 之间度量")
print("    隐含 %+.3f    局部 %+.3f    比值 %.2f" % (s_imp, s_loc, s_loc / s_imp))

例 21.1：一年期，S0 = 100
    Strike    Implied vol      Local vol
        80         23.37%         27.69%
        90         20.10%         21.82%
        95         18.49%         18.92%
       100         16.94%         16.15%
       105         15.53%         13.73%
       110         14.39%         12.03%
       120         13.33%         11.42%

  对数执行价上的斜率，在 95 与 105 之间度量
    隐含 -0.296    局部 -0.518    比值 1.75


## 5. 局部波动率为什么会把前向微笑定错：例 21.2

局部波动率函数*自身*的偏斜随期限衰减，比隐含偏斜快得多。
于是在这个模型下，一份三年后才起始的一年期期权，所用局部波动率的偏斜不到三个月期的四分之一，
模型据此预测的三年期前向微笑，会比今天的即期微笑平坦得多。

市场并不同意：远期起始期权与棘轮的成交偏斜与今天的相当。
错不在拟合（拟合是精确的），而在拟合所*蕴含*的动态。

In [5]:
print("例 21.2：各期限上在远期价格 +/-5% 处度量的偏斜")
print("  %-10s %14s %16s %10s" % ("Maturity", "Implied skew", "Local-vol skew", "Ratio"))
for T, lbl in [(0.25, "3 个月"), (0.5, "6 个月"), (1.0, "1 年"),
               (2.0, "2 年"), (3.0, "3 年")]:
    F = S0 * exp((R - Q) * T)
    lo, hi = F * 0.95, F * 1.05
    dk = log(hi / lo)
    si = (heston_iv(hi, T) - heston_iv(lo, T)) / dk
    sl = float(local_vol(hi, T, hT=min(0.005, T / 4))[0]
               - local_vol(lo, T, hT=min(0.005, T / 4))[0]) / dk
    print("  %-10s %14.3f %16.3f %10.2f" % (lbl, si, sl, sl / si))
print()
print("  局部波动率偏斜从 3 个月到 3 年衰减为原来的 1/%.1f"
      % (0.981 / 0.231))

例 21.2：各期限上在远期价格 +/-5% 处度量的偏斜
  Maturity     Implied skew   Local-vol skew      Ratio
  3 个月               -0.479           -0.981       2.05
  6 个月               -0.403           -0.770       1.91
  1 年                -0.296           -0.518       1.75
  2 年                -0.196           -0.320       1.63
  3 年                -0.148           -0.231       1.56

  局部波动率偏斜从 3 个月到 3 年衰减为原来的 1/4.2


## 6. 偏斜的期限结构：图 21.4

定义平值偏斜 $\psi(T) = -\partial\sigma_{\rm impl}/\partial\log K$（在 $K = F$ 处），
再看它在 $T \to 0$ 时的行为。

经验答案是一条幂律 $\psi(T) \sim T^{-0.4}$，适用于约一个月到数年的期限
（一个月以内的证据仍有争议，见 21.8 节）。
**Heston 再现不了这条幂律。** 方差是一个扩散过程，
在很短的时间里它只变动 $\sqrt{T}$ 量级；由此造成的不对称按 $\sqrt{T}$ 缩小，
度量偏斜所依据的分布宽度也按 $\sqrt{T}$ 缩小，所以无论参数怎么设，$\psi(T)$ 都趋于一个有限常数。

下面这个单元用 $\log\psi$ 对 $\log T$ 回归，量出 Heston 自己在短端的指数。
幂律会给出一条直线；Heston 给出的是一条正在压平的曲线，
拟合出的指数离 $-0.4$ 差得很远。

In [6]:
def atm_skew(T, h=0.01, **kw):
    F = S0 * exp((R - Q) * T)
    return -(heston_iv(F * exp(h), T, **kw) - heston_iv(F * exp(-h), T, **kw)) / (2 * h)


TS = np.array([1. / 12, 2. / 12, 0.25, 0.5, 1.0, 2.0, 3.0])
LBL = ["1 个月", "2 个月", "3 个月", "6 个月", "1 年", "2 年", "3 年"]
psi = np.array([atm_skew(T) for T in TS])
base = psi[TS == 1.0][0]

print("图 21.4：平值偏斜，以一年期归一")
print("  %-10s %12s %12s %12s" % ("Maturity", "Heston", "market", "gap"))
for lbl, T, p in zip(LBL, TS, psi):
    mkt = T ** -0.4
    print("  %-10s %12.3f %12.3f %11.0f%%" % (lbl, p / base, mkt, 100 * (p / base / mkt - 1)))

short = TS <= 0.5
slope = np.polyfit(np.log(TS[short]), np.log(psi[short]), 1)[0]
print()
print("  Heston 自己在短端的指数：%+.3f   （市场的是 -0.400）" % slope)
print("  从一年到一个月，Heston 升至 %.2f 倍；市场升至 %.2f 倍"
      % (psi[0] / base, (1. / 12) ** -0.4))

图 21.4：平值偏斜，以一年期归一
  Maturity         Heston       market          gap
  1 个月              1.773        2.702         -34%
  2 个月              1.744        2.048         -15%
  3 个月              1.665        1.741          -4%
  6 个月              1.381        1.320           5%
  1 年               1.000        1.000           0%
  2 年               0.656        0.758         -13%
  3 年               0.494        0.644         -23%

  Heston 自己在短端的指数：-0.138   （市场的是 -0.400）
  从一年到一个月，Heston 升至 1.77 倍；市场升至 2.70 倍


## 7. SABR：Hagan 展开

$$dF = \alpha F^\beta dW_1, \qquad d\alpha = \nu\alpha\,dW_2, \qquad dW_1dW_2 = \rho\,dt$$

让 SABR 成为利率市场标准的不是这个过程，而是那条渐近展开：
它直接给出*隐含波动率*，不用积分，不用模拟，一条微笑只需几微秒。

$\beta$ 决定**骨架**：远期价格变动时平值波动率如何变动。
$\beta = 1$ 时模型是对数正态的，$\beta = 0$ 时是正态的，
SABR 正是靠 $\beta = 0$ 处理负利率。
$\beta$ 通常是固定而不是拟合出来的，因为在单条微笑上它与 $\rho$ 几乎是冗余的。

已知的缺陷在最后一个单元里：在低执行价上，这条展开会蕴含一个**负密度**，
这时破坏 19.5 节无套利条件的是一个近似式，而不是市场。
检验办法是把 Breeden–Litzenberger 反着跑一遍：
给展开自己的微笑定价，作两次差分，看结果在哪里掉到零以下。

In [7]:
def sabr_vol(F, K, T, alpha, beta, rho, nu):
    """Hagan's 2002 lognormal-implied-volatility expansion."""
    one_b = 1.0 - beta
    fk = (F * K) ** (one_b / 2.0)
    lnfk = log(F / K)
    A = alpha / (fk * (1 + one_b ** 2 / 24 * lnfk ** 2 + one_b ** 4 / 1920 * lnfk ** 4))
    z = (nu / alpha) * fk * lnfk
    if abs(z) < 1e-12:
        zx = 1.0
    else:
        chi = log((sqrt(1 - 2 * rho * z + z * z) + z - rho) / (1 - rho))
        zx = z / chi
    B = 1 + (one_b ** 2 / 24 * alpha ** 2 / fk ** 2
             + rho * beta * nu * alpha / (4 * fk)
             + (2 - 3 * rho ** 2) / 24 * nu ** 2) * T
    return A * zx * B


F0, TSW = 0.030, 5.0
PAR = dict(alpha=0.030 ** 0.4 * 0.25, beta=0.6, rho=-0.30, nu=0.45)
strikes = np.array([0.010, 0.015, 0.020, 0.025, 0.030, 0.035, 0.040, 0.050, 0.060])

print("一条五年期互换期权微笑，F = 3.00%%，beta = %.1f" % PAR["beta"])
print("  %10s %14s" % ("Strike (%)", "Implied vol (%)"))
for K in strikes:
    print("  %10.2f %14.2f" % (100 * K, 100 * sabr_vol(F0, K, TSW, **PAR)))

print()
print("骨架：把远期价格挪动 50bp，看平值波动率如何变动")
print("  %-10s %14s %14s %14s" % ("beta", "F=2.50%", "F=3.00%", "F=3.50%"))
for beta in (0.0, 0.5, 1.0):
    a = 0.25 * F0 ** (1 - beta)                 # same ATM vol at F0 for every beta
    row = [100 * sabr_vol(f, f, TSW, a, beta, -0.30, 0.45) for f in (0.025, 0.030, 0.035)]
    print("  %-10.1f %14.2f %14.2f %14.2f" % (beta, row[0], row[1], row[2]))

print()
print("在展开自己的微笑上跑 Breeden-Litzenberger：密度在哪里变负？")
Kg = np.arange(0.0015, 0.0605, 0.00025)
prices = np.array([bs_call(F0, K, 0.0, 0.0, sabr_vol(F0, K, TSW, **PAR), TSW) for K in Kg])
dens = (prices[2:] - 2 * prices[1:-1] + prices[:-2]) / (Kg[1] - Kg[0]) ** 2
negs = Kg[1:-1][dens < 0]
if len(negs):
    print("  密度在执行价 %.3f%% 以下首次变负" % (100 * negs.max()))
    print("  最负的取值 %.1f，在执行价 %.3f%% 处"
          % (dens.min(), 100 * Kg[1:-1][dens.argmin()]))
else:
    print("  密度在整个网格上保持为正")

一条五年期互换期权微笑，F = 3.00%，beta = 0.6
  Strike (%) Implied vol (%)
        1.00          45.56
        1.50          37.89
        2.00          32.60
        2.50          28.82
        3.00          26.24
        3.50          24.69
        4.00          23.98
        5.00          24.01
        6.00          24.83

骨架：把远期价格挪动 50bp，看平值波动率如何变动
  beta              F=2.50%        F=3.00%        F=3.50%
  0.0                 32.75          27.15          23.20
  0.5                 28.86          26.38          24.45
  1.0                 25.77          25.77          25.77

在展开自己的微笑上跑 Breeden-Litzenberger：密度在哪里变负？
  密度在执行价 0.250% 以下首次变负
  最负的取值 -3.4，在执行价 0.175% 处


## 8. 跳跃：例 21.3

Merton 模型是扩散加复合泊松跳跃，价格是一列泊松加权的布莱克–斯科尔斯–默顿价格之和，

$$C = \sum_{n\ge 0} \frac{e^{-\lambda^*T}(\lambda^*T)^n}{n!}\,
      C_{\rm BS}(S,K,r_n,\sigma_n,T)$$

其中 $\sigma_n^2 = \sigma^2 + n\delta^2/T$，漂移则作了补偿，使贴现价格仍是鞅。

跳跃能做到扩散做不到的事：在一个月的时间里，
连续路径根本走不了那么远，不足以让一个虚值 15% 的看跌期权值钱，
只有不连续才能。正因如此，模型能生成陡峭的短期微笑；
微笑随期限收窄，则是因为扩散的幅度随期限增大，而跳跃的幅度不变。

而这同时也是对它的批评。**跳跃在短端偏陡**：
以一个月与一年之间的平值偏斜之比来衡量，模型给出 3.7，比股指偏斜所遵循的幂律给出的 2.7 高出三分之一以上。

In [8]:
def merton_call(K, T, sig=0.17, lam=0.5, kbar=-0.10, delJ=0.13, nmax=60):
    """kbar = E[Y] - 1 is the mean jump; muJ follows from it."""
    muJ = log(1 + kbar) - 0.5 * delJ ** 2
    # The sum discounts at r while each term discounts at its own
    # r_n; reconciling the two compensates the intensity.
    lam_star = lam * (1 + kbar)
    tot, lamT = 0.0, lam_star * T
    for n in range(nmax):
        sig_n = sqrt(sig ** 2 + n * delJ ** 2 / T)
        r_n = R - lam * kbar + n * (muJ + 0.5 * delJ ** 2) / T
        tot += exp(-lamT) * lamT ** n / factorial(n) * bs_call(S0, K, r_n, Q, sig_n, T)
    return tot


def merton_iv(K, T, **kw):
    return implied_vol(merton_call(K, T, **kw), S0, K, R, Q, T)


TS3 = [1. / 12, 0.25, 1.0]
KS3 = [85., 90., 95., 100., 105., 110.]
print("例 21.3：sigma = 17%，lambda = 0.5，平均跳幅 -10%，跳跃波动率 13%")
print("  %8s %12s %12s %12s" % ("Strike", "1 month", "3 months", "1 year"))
tab = {}
for K in KS3:
    row = [100 * merton_iv(K, T) for T in TS3]
    tab[K] = row
    print("  %8.0f %11.2f%% %11.2f%% %11.2f%%" % (K, row[0], row[1], row[2]))

m1 = [tab[K][0] for K in KS3]
y1 = [tab[K][2] for K in KS3]
print()
print("  一个月：从 %.1f%% 降到 %.1f%%，跨度 %.1f 个波动率点"
      % (max(m1), min(m1), max(m1) - min(m1)))
print("  一年：  从 %.1f%% 降到 %.1f%%，跨度 %.1f 个波动率点"
      % (max(y1), min(y1), max(y1) - min(y1)))
print("  跨度缩小到原来的 1/%.1f" % ((max(m1) - min(m1)) / (max(y1) - min(y1))))


def merton_skew(T, h=0.01):
    F = S0 * exp((R - Q) * T)
    return -(merton_iv(F * exp(h), T) - merton_iv(F * exp(-h), T)) / (2 * h)


r_model = merton_skew(1. / 12) / merton_skew(1.0)
print()
print("  平值偏斜之比，一个月比一年：模型 %.1f，市场约 2.7" % r_model)
print("  跳跃在短端的结果是市场的 %.1f 倍" % (r_model / 2.7))

例 21.3：sigma = 17%，lambda = 0.5，平均跳幅 -10%，跳跃波动率 13%
    Strike      1 month     3 months       1 year


        85       30.99%       23.82%       21.21%
        90       25.07%       21.78%       20.81%
        95       20.57%       20.35%       20.47%


       100       18.77%       19.49%       20.18%
       105       18.25%       18.99%       19.94%


       110       18.42%       18.73%       19.74%

  一个月：从 31.0% 降到 18.2%，跨度 12.7 个波动率点
  一年：  从 21.2% 降到 19.7%，跨度 1.5 个波动率点
  跨度缩小到原来的 1/8.7

  平值偏斜之比，一个月比一年：模型 3.7，市场约 2.7
  跳跃在短端的结果是市场的 1.4 倍


## 9. 粗糙波动率：式 (21.8) 与 (21.9)

对数波动率的行为像一个 Hurst 参数 $H \approx 0.1$ 的分数布朗运动，
这样的模型会生成

$$\psi(T) \sim T^{H - 1/2}$$

于是 $H = 0.1$ 恰好给出观测到的 $T^{-0.4}$，只用一个参数；
而 $H = 0.5$ 则回到经典模型那种平坦的短端。

难点在计算上：分数布朗运动不是马尔可夫的，
于是每个价格都得模拟，而精确模拟的代价是 $O(n^2)$。
下面这个**混合格式**正是改变了这一点的进展之一：
它把核的奇异部分精确处理（第一项，与布朗增量按一个 $2\times2$ 协方差联合抽取），
其余光滑部分用卷积，这里直接用 `np.convolve` 计算；改用 FFT 卷积可降到 $O(n\log n)$，
正是这一点使该格式能与普通蒙特卡洛相比。

表 21.4 接着从基准情形出发逐一变动三个参数，并像表 21.2 对 Heston 所做的那样，从模拟出的微笑上读出一个月与一年的水平、偏斜与曲率。

检验仍是开创这一领域的那项检验：在多个期限上量出 $\psi(T)$，
用 $\log\psi$ 对 $\log T$ 回归，看斜率是不是回到 $H - 1/2$。
与第 6 节对照：那里 Heston 的斜率差得很远。

In [9]:
def rbergomi(H, eta, rho, xi0, T, n_steps, n_paths, seed=5):
    """Bennedsen-Lunde-Pakkanen hybrid scheme, kappa = 1."""
    dt = T / n_steps
    a = H - 0.5
    rng = np.random.default_rng(seed)
    cov = np.array([[dt, dt ** (a + 1) / (a + 1)],
                    [dt ** (a + 1) / (a + 1), dt ** (2 * a + 1) / (2 * a + 1)]])
    inc = rng.standard_normal((n_paths, n_steps, 2)) @ np.linalg.cholesky(cov).T
    dW1, Wa = inc[:, :, 0], inc[:, :, 1]
    kk = np.arange(2, n_steps + 1)
    b = ((kk ** (a + 1) - (kk - 1) ** (a + 1)) / (a + 1)) ** (1.0 / a)
    G = np.zeros(n_steps)
    G[1:] = (b * dt) ** a
    Y = sqrt(2 * H) * (Wa + np.array([np.convolve(r, G)[:n_steps] for r in dW1]))
    t = np.arange(1, n_steps + 1) * dt
    V = xi0 * np.exp(eta * Y - 0.5 * eta ** 2 * t ** (2 * H))
    dB = rho * dW1 + sqrt(1 - rho ** 2) * rng.standard_normal((n_paths, n_steps)) * sqrt(dt)
    Vp = np.concatenate([np.full((n_paths, 1), xi0), V[:, :-1]], axis=1)
    return S0 * np.exp(np.cumsum(-0.5 * Vp * dt + np.sqrt(Vp) * dB, axis=1))[:, -1]


def mc_skew(ST, T, w=0.05):
    f = lambda K: implied_vol(np.maximum(ST - K, 0).mean(), S0, K, R, Q, T)
    return -(f(S0 * exp(w)) - f(S0 * exp(-w))) / (2 * w)


MATS = np.array([1. / 12, 2. / 12, 0.25, 0.5, 1.0])
print("粗糙 Bergomi：eta = 1.9，rho = -0.9，xi0 = 0.04，15 万条路径")
print("  %-6s %s %14s" % ("H", "".join("%11s" % L for L in
                                        ["1m", "2m", "3m", "6m", "1y"]), "fitted slope"))
for H in (0.10, 0.25, 0.49):
    row = []
    for T in MATS:
        n = max(int(round(252 * T)), 21)
        row.append(mc_skew(rbergomi(H, 1.9, -0.9, 0.04, T, n, 150_000), T))
    slope = np.polyfit(np.log(MATS), np.log(row), 1)[0]
    print("  %-6.2f %s %14.3f   (H - 1/2 = %+.2f)"
          % (H, "".join("%11.3f" % x for x in row), slope, H - 0.5))
print()
print("  一个参数就复现了观测到的幂律；Heston 的短端斜率在第 6 节量过，")
print("  与这里任何一个都不接近。")

def mc_stats(ST, T, h=0.10):
    """ATM level, skew and curvature from a +-h central difference in log-moneyness."""
    lo, mid, hi = [implied_vol(np.maximum(ST - K, 0).mean(), S0, K, R, Q, T)
                   for K in (S0 * exp(-h), S0, S0 * exp(h))]
    return mid, (hi - lo) / (2 * h), (hi - 2 * mid + lo) / h ** 2


BASE_RB = dict(H=0.10, eta=1.9, rho=-0.9, xi0=0.04)
ROWS_RB = [("base", {}), ("H = 0.25", dict(H=0.25)), ("H = 0.45", dict(H=0.45)),
           ("eta = 1.0", dict(eta=1.0)), ("eta = 3.0", dict(eta=3.0)),
           ("rho = -0.5", dict(rho=-0.5)), ("rho = 0", dict(rho=0.0))]
print()
print("表 21.4：各粗糙 Bergomi 参数移动什么（200,000 条路径，种子 5）")
print("  %-11s %-22s   %-22s   %s" % ("", "1m: ATM   skew  curv", "1y: ATM   skew  curv", "1m/1y skew"))
for name, change in ROWS_RB:
    p = dict(BASE_RB, **change)
    rows = []
    for T, n in ((1. / 12, 21), (1.0, 252)):
        ST = rbergomi(p["H"], p["eta"], p["rho"], p["xi0"], T, n, 200_000)
        rows.append((mc_stats(ST, T), mc_skew(ST, T)))
    (a1, s1), (a12, s12) = rows
    ratio = "%5.1f" % (s1 / s12) if abs(s12) > 0.01 else "  ---"
    print("  %-11s %5.1f %6.2f %5.1f       %5.1f %6.2f %5.1f       %s"
          % (name, 100 * a1[0], round(a1[1], 2) + 0.0, a1[2], 100 * a12[0], round(a12[1], 2) + 0.0, a12[2], ratio))
print()
print("  H 移动的是 1m/1y 比值（理论值 12**(0.5 - H)：2.70、1.86、1.13），而一年偏斜几乎不动。")


粗糙 Bergomi：eta = 1.9，rho = -0.9，xi0 = 0.04，15 万条路径
  H               1m         2m         3m         6m         1y   fitted slope


  0.10         0.815      0.669      0.561      0.431      0.321         -0.380   (H - 1/2 = -0.40)


  0.25         0.775      0.674      0.586      0.484      0.390         -0.281   (H - 1/2 = -0.25)


  0.49         0.416      0.431      0.407      0.391      0.357         -0.066   (H - 1/2 = -0.01)

  一个参数就复现了观测到的幂律；Heston 的短端斜率在第 6 节量过，
  与这里任何一个都不接近。

表 21.4：各粗糙 Bergomi 参数移动什么（200,000 条路径，种子 5）
              1m: ATM   skew  curv     1y: ATM   skew  curv     1m/1y skew


  base         17.8  -0.63   4.3        16.3  -0.31   0.0         2.6


  H = 0.25     18.6  -0.73   0.1        15.1  -0.39  -0.3         2.0


  H = 0.45     19.5  -0.47  -0.3        15.3  -0.36  -0.2         1.3


  eta = 1.0    19.3  -0.51  -0.1        18.6  -0.19  -0.1         2.7


  eta = 3.0    15.2  -0.47  13.3        12.8  -0.30   2.1         2.1


  rho = -0.5   18.3  -0.34   5.9        17.1  -0.17   0.9         2.4


  rho = 0      18.6   0.00   6.7        17.7   0.00   1.2         ---

  H 移动的是 1m/1y 比值（理论值 12**(0.5 - H)：2.70、1.86、1.13），而一年偏斜几乎不动。


## 10. 局部–随机波动率：例 21.5 与图 21.6

奇异期权交易台真正在跑的是两个模型合在一起，

$$dS = (r-q)S\,dt + L(S,t)\sqrt{v}\,S\,dW_1$$

其中 $L$ 就是**杠杆函数**，选它是为了让普通期权的拟合精确。定义式是

$$L^2(x,t) = \frac{\sigma^2_{\rm loc}(x,t)}{\mathbb{E}[v_t \mid S_t = x]}$$

这是循环的：条件期望依赖 $L$，而 $L$ 又由它定义。
**粒子法**打破这个循环的办法，是在模拟路径向前推进的同时，
用这些路径自己估计这一条件期望。这里按对数即期价格的分位数分箱，
是能用的最简陋的估计量。

**混合参数**缩放随机部分：取 0 时模型是纯局部波动率，
取 1 时则在精确拟合所允许的范围内尽量接近纯随机波动率。
检验在于普通期权的微笑在*每一档设定下*都必须复现，
因为杠杆函数吸收掉随机部分没能解释的全部差异。

In [10]:
T_GRID = np.concatenate([np.linspace(0.02, 0.20, 19), np.linspace(0.22, 1.06, 43)])
LOGK = np.linspace(log(20.0), log(400.0), 161)
SURF = np.array([local_vol(np.exp(LOGK), T) for T in T_GRID])
print("局部波动率曲面：%d 个期限 x %d 个执行价" % SURF.shape)
print("  sigma_loc(100, 1y) = %.2f%%   sigma_loc(110, 1y) = %.2f%%"
      % (100 * np.interp(log(100), LOGK, SURF[-4]),
         100 * np.interp(log(110), LOGK, SURF[-4])))


def lv_lookup(S, t):
    """Bilinear in (t, log S), flat outside the grid."""
    i = min(max(np.searchsorted(T_GRID, t) - 1, 0), len(T_GRID) - 2)
    w = min(max((t - T_GRID[i]) / (T_GRID[i + 1] - T_GRID[i]), 0.0), 1.0)
    row = (1 - w) * SURF[i] + w * SURF[i + 1]
    return np.interp(np.log(np.clip(S, exp(LOGK[0]), exp(LOGK[-1]))), LOGK, row)


LEVERAGE_SLICE = {}   # mixing -> (terminal spot, L on the final slice), filled by lsv_paths


def lsv_paths(mix, n_paths=80_000, n_steps=252, T=1.0, nbin=60, seed=99, up=110.0):
    """Particle-method LSV. mix = 0 is pure local volatility, mix = 1 pure stochastic.

    Returns the terminal spot and the survival mask of an up-and-out at `up`.
    """
    dt = T / n_steps
    rng = np.random.default_rng(seed)
    S = np.full(n_paths, S0)
    v = np.full(n_paths, BASE["v0"])
    alive = np.ones(n_paths, bool)
    for i in range(n_steps):
        vp = np.maximum(v, 0.0)
        loc = lv_lookup(S, i * dt)
        if mix == 0.0:
            sig = loc
            if i == n_steps - 1:
                LEVERAGE_SLICE[mix] = (S.copy(), loc / sqrt(BASE["v0"]))
        else:
            y = np.log(S)
            edges = np.quantile(y, np.linspace(0, 1, nbin + 1))
            edges[0] -= 1e-9
            edges[-1] += 1e-9
            idx = np.clip(np.searchsorted(edges, y) - 1, 0, nbin - 1)
            cnt = np.bincount(idx, minlength=nbin)
            cond = np.bincount(idx, weights=vp, minlength=nbin) / np.maximum(cnt, 1)
            cond = np.where(cnt > 0, cond, vp.mean())
            sig = loc * np.sqrt(vp / np.maximum(cond[idx], 1e-8))    # L * sqrt(v)
            if i == n_steps - 1:
                LEVERAGE_SLICE[mix] = (S.copy(), loc / np.sqrt(np.maximum(cond[idx], 1e-8)))
        z1 = rng.standard_normal(n_paths)
        z2 = BASE["rho"] * z1 + sqrt(1 - BASE["rho"] ** 2) * rng.standard_normal(n_paths)
        S = S * np.exp(-0.5 * sig ** 2 * dt + sig * sqrt(dt) * z1)
        alive &= S < up
        v = v + BASE["kappa"] * (BASE["theta"] - vp) * dt + \
            mix * BASE["sigv"] * np.sqrt(vp * dt) * z2
    return S, alive


print()
print("杠杆函数在每一档混合水平上都把普通期权的拟合还原了吗？")
print("  %-14s %s" % ("mixing", "".join("%14s" % ("K=%d" % K) for K in (80, 100, 120))))
tgt = [100 * heston_iv(K, 1.0) for K in (80., 100., 120.)]
for mix in (0.0, 0.25, 0.5, 0.75, 1.0):
    ST = lsv_paths(mix)[0]
    row = [100 * implied_vol(np.maximum(ST - K, 0).mean(), S0, K, R, Q, 1.0)
           for K in (80., 100., 120.)]
    print("  %-14.2f %s" % (mix, "".join("%13.2f%%" % x for x in row)))
print("  %-14s %s" % ("target", "".join("%13.2f%%" % x for x in tgt)))

print()
print("图 21.6：一年处的杠杆函数，按即期区间取中位数")
BUCKETS = [70, 80, 90, 100, 110, 120, 130]
print("  %-8s %s" % ("mixing", "".join("%9s" % ("%d-%d" % (lo, hi)) for lo, hi in zip(BUCKETS[:-1], BUCKETS[1:]))))
for mix in (0.0, 0.5, 1.0):
    Sfin, Lfin = LEVERAGE_SLICE[mix]
    prof = [np.median(Lfin[(Sfin >= lo) & (Sfin < hi)]) for lo, hi in zip(BUCKETS[:-1], BUCKETS[1:])]
    print("  %-8.1f %s" % (mix, "".join("%9.3f" % x for x in prof)))
print("  混合参数为 1 时曲面就是模型自己的，于是算出的 L 处处等于一。")

局部波动率曲面：62 个期限 x 161 个执行价
  sigma_loc(100, 1y) = 16.15%   sigma_loc(110, 1y) = 12.03%

杠杆函数在每一档混合水平上都把普通期权的拟合还原了吗？
  mixing                   K=80         K=100         K=120


  0.00                   23.35%        16.96%        13.30%


  0.25                   23.34%        16.94%        13.29%


  0.50                   23.27%        16.89%        13.28%


  0.75                   23.21%        16.85%        13.23%


  1.00                   23.24%        16.85%        13.23%
  target                 23.37%        16.94%        13.33%

图 21.6：一年处的杠杆函数，按即期区间取中位数
  mixing       70-80    80-90   90-100  100-110  110-120  120-130
  0.0          1.509    1.220    0.924    0.681    0.570    0.588
  0.5          1.153    1.009    0.876    0.774    0.768    0.870
  1.0          1.005    1.001    0.994    1.008    0.991    0.979
  混合参数为 1 时曲面就是模型自己的，于是算出的 L 处处等于一。


## 11. 把模型风险做实：例 21.7

从 Heston *自己*的普通期权价格里导出局部波动率曲面。
这样两个模型在构造上就对每一份欧式期权都一致（下一个单元会确认这一点），
于是障碍期权上的任何差异都是纯粹的模型风险。

这一比较之所以可信，靠的是两点做法。两个模型在即期价格的布朗运动上用*同一组随机数*，
于是差异是逐路径度量的，而不是两个含噪价格相减；
这正是 20.10 节引入那项技术的目的。
其次，报出的标准误差是*差值的*标准误差，它比任何一个价格自身的标准误差小一个数量级。

结果就是这张表两半之间的反差。对向下敲出期权，两个模型相差不到百分之一点五；
对反向敲出期权，两者相差将近价格的一半。

*所以模型风险不是“障碍期权”的属性。*
它取决于收益所在的位置与两个模型分歧所在位置的相对关系。
有一个听上去很有道理的论证：局部波动率在高即期价格处*低*，所以它的路径更少敲出。
这个前提本身就不成立，它的路径敲出得更多。
两个模型给出的终端价格分布相同，关键在于给定终点时的存活概率：
Heston 下有不触及 110 也能到达窄带的平静路径，局部波动率对每个即期价格与日期只给出一个波动率，没有这样的路径。
最后一个单元逐区间印出存活概率。

In [11]:
N_PATHS, N_STEPS, TB = 200_000, 252, 1.0
DT = TB / N_STEPS
rng = np.random.default_rng(20250815)
Z1 = rng.standard_normal((N_STEPS, N_PATHS // 2))
Z1 = np.concatenate([Z1, -Z1], axis=1)                 # antithetic
Z3 = rng.standard_normal((N_STEPS, N_PATHS // 2))
Z3 = np.concatenate([Z3, -Z3], axis=1)

DOWN, UP = [80.0, 85.0, 90.0], [130.0, 120.0, 115.0, 110.0]


def barrier_run(model):
    """Daily monitoring. Returns terminal spot and the survival masks."""
    S = np.full(N_PATHS, S0)
    v = np.full(N_PATHS, BASE["v0"])
    alive = {("do", b): np.ones(N_PATHS, bool) for b in DOWN}
    alive.update({("uo", b): np.ones(N_PATHS, bool) for b in UP})
    for i in range(N_STEPS):
        if model == "heston":
            vp = np.maximum(v, 0.0)
            sig = np.sqrt(vp)
            z2 = BASE["rho"] * Z1[i] + sqrt(1 - BASE["rho"] ** 2) * Z3[i]
            v = v + BASE["kappa"] * (BASE["theta"] - vp) * DT \
                + BASE["sigv"] * np.sqrt(vp * DT) * z2
        else:
            sig = lv_lookup(S, i * DT)
        S = S * np.exp(-0.5 * sig ** 2 * DT + sig * sqrt(DT) * Z1[i])
        for b in DOWN:
            alive[("do", b)] &= S > b
        for b in UP:
            alive[("uo", b)] &= S < b
    return S, alive


res = {m: barrier_run(m) for m in ("heston", "localvol")}

print("普通期权对照：两个模型在一年期 100 执行价看涨上必须一致")
for m in ("heston", "localvol"):
    pay = np.maximum(res[m][0] - 100.0, 0.0)
    print("  %-9s %8.4f +/- %.4f" % (m, pay.mean(), pay.std(ddof=1) / sqrt(N_PATHS)))
print("  %-9s %8.4f  (半闭式解)" % ("Heston", heston_call(100.0, 1.0)[0]))

print()
print("例 21.7：一年期，每日监测，共同随机数")
print("  %-10s %10s %10s %18s %10s" % ("Barrier", "Heston", "Local vol", "Difference", "Relative"))
for kind, bars, label in (("do", DOWN, "向下敲出看涨，执行价 100"),
                          ("uo", UP, "向上敲出看涨，执行价 100")):
    print("  %s" % label)
    for b in bars:
        ph = np.maximum(res["heston"][0] - 100.0, 0.0) * res["heston"][1][(kind, b)]
        pl = np.maximum(res["localvol"][0] - 100.0, 0.0) * res["localvol"][1][(kind, b)]
        d = pl - ph
        print("     %-7.0f %10.4f %10.4f   %+8.3f +/- %.3f %9.1f%%"
              % (b, ph.mean(), pl.mean(), d.mean(),
                 d.std(ddof=1) / sqrt(N_PATHS), 100 * d.mean() / ph.mean()))

普通期权对照：两个模型在一年期 100 执行价看涨上必须一致
  heston      6.7655 +/- 0.0192
  localvol    6.7669 +/- 0.0194
  Heston      6.7509  (半闭式解)

例 21.7：一年期，每日监测，共同随机数
  Barrier        Heston  Local vol         Difference   Relative
  向下敲出看涨，执行价 100
     80          6.5286     6.5821     +0.053 +/- 0.015       0.8%
     85          6.2180     6.2962     +0.078 +/- 0.015       1.3%
     90          5.5388     5.6138     +0.075 +/- 0.015       1.4%
  向上敲出看涨，执行价 100
     130         5.8138     5.5787     -0.235 +/- 0.015      -4.0%
     120         4.0046     3.4193     -0.585 +/- 0.013     -14.6%
     115         2.4071     1.7138     -0.693 +/- 0.010     -28.8%
     110         0.8402     0.4244     -0.416 +/- 0.005     -49.5%


In [12]:
print("向上敲出为什么是这个方向：给定终点所在区间的存活概率")
print("  %-10s %10s %16s %16s" % ("S_T band", "P(band)", "P(alive|band) H", "P(alive|band) LV"))
for lo, hi in ((100, 102), (102, 104), (104, 106), (106, 108), (108, 110)):
    out = []
    for m in ("heston", "localvol"):
        ST, alive = res[m]
        inb = (ST >= lo) & (ST < hi)
        out.append((inb.mean(), alive[("uo", 110.0)][inb].mean()))
    print("  %3d-%-6d %9.1f%% %15.1f%% %15.1f%%"
          % (lo, hi, 100 * out[0][0], 100 * out[0][1], 100 * out[1][1]))
for m in ("heston", "localvol"):
    ST, alive = res[m]
    a = alive[("uo", 110.0)] & (ST > 100.0)
    print("  %-9s P(alive, S_T > 100) = %5.1f%%   E[payoff | alive, S_T > 100] = %.2f"
          % (m, 100 * a.mean(), (ST[a] - 100.0).mean()))
print()
print("  两个模型在每个区间上的概率相同，但终点落在区间内的局部波动率路径，")
print("  途中触及 110 的比例要高得多。")

print()
print("例 21.5：第 10 节的混合参数，用在 110 向上敲出上")
print("  %-10s %14s %16s %14s %12s" % ("mixing", "ATM check", "up-and-out 110", "std error", "P(survive)"))
rng_lo, rng_hi = None, None
for mix in (0.0, 0.25, 0.5, 0.75, 1.0):
    ST, alive = lsv_paths(mix)
    pay = np.maximum(ST - 100.0, 0.0) * alive
    atm = 100 * implied_vol(np.maximum(ST - 100.0, 0).mean(), S0, 100.0, R, Q, 1.0)
    print("  %-10.2f %13.2f%% %16.4f %14.4f %11.1f%%"
          % (mix, atm, pay.mean(), pay.std(ddof=1) / sqrt(len(pay)), 100 * alive.mean()))
    rng_lo = pay.mean() if rng_lo is None else min(rng_lo, pay.mean())
    rng_hi = pay.mean() if rng_hi is None else max(rng_hi, pay.mean())
print("  区间 %.3f 到 %.3f（相差 %.1f 倍）就是应当报告给风险委员会的数字，"
      % (rng_lo, rng_hi, rng_hi / rng_lo))
print("  而这个旋钮的意义在于，它出自同一个模型。")

向上敲出为什么是这个方向：给定终点所在区间的存活概率
  S_T band      P(band)  P(alive|band) H P(alive|band) LV
  100-102          5.1%            76.7%            56.9%
  102-104          5.6%            73.4%            49.4%
  104-106          6.0%            68.3%            39.7%
  106-108          6.3%            59.2%            28.4%
  108-110          6.0%            39.3%            13.6%
  heston    P(alive, S_T > 100) =  18.3%   E[payoff | alive, S_T > 100] = 4.60
  localvol  P(alive, S_T > 100) =  10.7%   E[payoff | alive, S_T > 100] = 3.96

  两个模型在每个区间上的概率相同，但终点落在区间内的局部波动率路径，
  途中触及 110 的比例要高得多。

例 21.5：第 10 节的混合参数，用在 110 向上敲出上
  mixing          ATM check   up-and-out 110      std error   P(survive)


  0.00               16.96%           0.4202         0.0052        44.8%


  0.25               16.94%           0.4628         0.0055        45.7%


  0.50               16.89%           0.5724         0.0061        47.7%


  0.75               16.85%           0.7074         0.0068        50.2%


  1.00               16.85%           0.8449         0.0075        52.7%
  区间 0.420 到 0.845（相差 2.0 倍）就是应当报告给风险委员会的数字，
  而这个旋钮的意义在于，它出自同一个模型。


## 12. 校准目标函数：例 21.6

对同一曲面在不同目标函数下作 Heston 校准：**价格**误差平方和、**隐含波动率**误差平方和，
以及**除以 vega** 后的价格误差平方和。后两者在一阶上是同一个目标函数，
因为 $dC = \mathrm{vega}\times d\sigma$，它们正是 21.10 节推荐的做法。

这里的曲面由第 8 节的跳跃模型生成，所以任何 Heston 拟合都不可能精确。
这才是如实的设定：在一个模型本来就能复现的曲面上作拟合，关于目标函数什么也说明不了。

价格误差由平值期权主导，因为它们的 vega 最大，同样的波动率误差在那里代价最高。
这一曲面上一个月期执行价 80 的看跌期权只值 \$0.05，价格拟合让它的隐含波动率低了 7.08 个点。

这个单元还运行了 21.10 节提醒不要用的目标函数：*乘以* vega 的波动率误差。
它几乎精确地复现了价格误差的拟合，而且必然如此，
因为按 vega 加权的波动率误差在一阶上就是价格误差。
按 vega 加权救不回两翼；除以 vega，或者直接拟合波动率，才能把误差从两翼挪走。

它们都不是普遍正确的。价格目标适合以平值跨式为主的账簿，
波动率目标适合卖掉了两翼的账簿。目标函数隐含着关于账簿的假定，
应当有意识地选定，而不是沿用软件的默认设置。

In [13]:
KS_CAL = np.array([80., 85., 90., 95., 100., 105., 110., 115., 120.])
TS_CAL = np.array([1. / 12, 0.25, 1.0])
MKT_P = np.array([[merton_call(K, T) for K in KS_CAL] for T in TS_CAL])
MKT_V = np.array([[implied_vol(MKT_P[i, j], S0, KS_CAL[j], R, Q, TS_CAL[i])
                   for j in range(len(KS_CAL))] for i in range(len(TS_CAL))])
VEGA = np.array([[bs_vega(S0, KS_CAL[j], R, Q, MKT_V[i, j], TS_CAL[i])
                  for j in range(len(KS_CAL))] for i in range(len(TS_CAL))])


def model_grid(x):
    v0, kappa, theta, sigv, rho = x
    p = dict(v0=v0, kappa=kappa, theta=theta, sigv=sigv, rho=rho)
    return np.array([heston_call(KS_CAL, T, **p) for T in TS_CAL])


X0 = [0.04, 1.5, 0.04, 0.6, -0.5]
LO = [0.005, 0.10, 0.005, 0.05, -0.99]
HI = [0.50, 15.0, 0.50, 3.0, 0.50]


def vol_grid(x):
    P = model_grid(x)
    return np.array([[implied_vol(max(P[i, j], 1e-12), S0, KS_CAL[j], R, Q, TS_CAL[i])
                      for j in range(len(KS_CAL))] for i in range(len(TS_CAL))])


def resid_price(x):
    return (model_grid(x) - MKT_P).ravel()


def resid_vega(x):
    return ((vol_grid(x) - MKT_V) * VEGA).ravel()


def resid_flat(x):
    return (vol_grid(x) - MKT_V).ravel()


def resid_pdv(x):
    return ((model_grid(x) - MKT_P) / VEGA).ravel()


fits = {}
for tag, f in (("price errors", resid_price),
               ("vega-weighted vol errors", resid_vega),
               ("equally weighted vol errors", resid_flat),
               ("price errors / vega", resid_pdv)):
    s = least_squares(f, X0, bounds=(LO, HI), xtol=1e-10, ftol=1e-10)
    fits[tag] = s.x
    print("  %-29s v0=%.4f kappa=%.2f theta=%.4f sigv=%.3f rho=%+.3f"
          % (tag, s.x[0], s.x[1], s.x[2], s.x[3], s.x[4]))
print("  每一种拟合都把 kappa 顶到了边界：Heston 被要求复现一条只有跳跃才造得出的")
print("  短期微笑，而快速均值回复是它能做到的最接近的近似。参数顶在边界上是一个信号，")
print("  不是一个麻烦。")

print()
print("隐含波动率误差，单位为波动率点，一个月期，即各种拟合分道扬镳的地方")
print("  %-29s %s" % ("", "".join("%8.0f" % K for K in KS_CAL)))
for tag in fits:
    err = 100 * (vol_grid(fits[tag])[0] - MKT_V[0])
    print("  %-29s %s" % (tag, "".join("%8.2f" % e for e in err)))
    print("  %-29s   平值 %+.2f，最差的翼 %+.2f，平均绝对值 %.2f"
          % ("", err[4], max(list(err[:2]) + list(err[-2:]), key=abs), np.mean(np.abs(err))))
print()
print("前两种拟合几乎重合，而这正是结论，不是巧合：")
print("  按 vega 加权的波动率误差在一阶上就是价格误差，因为 dC = vega x d(sigma)，")
print("  所以按 vega 加权救不回两翼。把价格误差除以 vega，即最后一种拟合，")
print("  才是 21.10 节推荐的目标函数：一阶上它就是等权的波动率拟合，")
print("  以波动率点度量每一个误差，")
print("  并把误差从两翼挪到接近平值的执行价上，")
print("  这正是以两翼为主的账簿想要的取舍。")

  price errors                  v0=0.0381 kappa=15.00 theta=0.0432 sigv=1.308 rho=-0.298


  vega-weighted vol errors      v0=0.0389 kappa=15.00 theta=0.0433 sigv=1.383 rho=-0.288


  equally weighted vol errors   v0=0.0414 kappa=15.00 theta=0.0446 sigv=1.915 rho=-0.386
  price errors / vega           v0=0.0403 kappa=15.00 theta=0.0442 sigv=1.724 rho=-0.378
  每一种拟合都把 kappa 顶到了边界：Heston 被要求复现一条只有跳跃才造得出的
  短期微笑，而快速均值回复是它能做到的最接近的近似。参数顶在边界上是一个信号，
  不是一个麻烦。

隐含波动率误差，单位为波动率点，一个月期，即各种拟合分道扬镳的地方
                                      80      85      90      95     100     105     110     115     120
  price errors                     -7.08   -4.51   -1.27    0.53    0.12    0.04    0.73    0.73   -0.48
                                  平值 +0.12，最差的翼 -7.08，平均绝对值 1.72


  vega-weighted vol errors         -6.67   -4.20   -1.06    0.62    0.13    0.12    0.96    1.08   -0.04
                                  平值 +0.13，最差的翼 -6.67，平均绝对值 1.65
  equally weighted vol errors      -2.94   -1.26    0.94    1.49   -0.32   -0.68    0.68    1.24    0.41
                                  平值 -0.32，最差的翼 -2.94，平均绝对值 1.11
  price errors / vega              -4.01   -2.07    0.41    1.28   -0.19   -0.59    0.51    0.88   -0.06
                                  平值 -0.19，最差的翼 -4.01，平均绝对值 1.11

前两种拟合几乎重合，而这正是结论，不是巧合：
  按 vega 加权的波动率误差在一阶上就是价格误差，因为 dC = vega x d(sigma)，
  所以按 vega 加权救不回两翼。把价格误差除以 vega，即最后一种拟合，
  才是 21.10 节推荐的目标函数：一阶上它就是等权的波动率拟合，
  以波动率点度量每一个误差，
  并把误差从两翼挪到接近平值的执行价上，
  这正是以两翼为主的账簿想要的取舍。


## 这个 notebook 说明了什么

* Heston 的特征函数精确到机器精度，这正是 Dupire 的二阶导数能用有限差分求出来的前提。
  任何基于模拟的曲面都会留下太多噪声，根本做不了这件事。
* **二倍法则**是数学上的必然结果，不是校准失败：局部波动率的陡度大约是由它导出的隐含波动率的两倍。
* 局部波动率的拟合是精确的，而它的*前向*微笑是错的。
  这两件事并不矛盾：拟合约束的是终端分布，而前向微笑不在其中。
* Heston 的短端偏斜会饱和，市场的不会（至少到一个月为止）。跳跃则矫枉过正。
  粗糙波动率只用一个参数就拟合出了这一指数，所以它备受关注；
  波动率是否真的粗糙，仍有争论。
* **模型风险是收益结构的属性，不是工具类别的属性。**
  两个拟合了同一批普通期权的模型，对向下敲出期权相差不到 1.5%，
  对反向敲出期权却相差将近一半的价格。

所有结果都能从本章印出的参数复现。凡本处的数字是蒙特卡洛估计的，
其标准误差都印在旁边，一致性应当按这个单位来读。